# Synthesize data for LLM Training/Fine-tuning

In [ ]:
%pip install "data-designer>=0.6.0" "deepeval>=3.8.3" "ipykernel>=7.2.0" "langchain>=1.2.7" "langchain-community>=0.4.1" "langchain-core>=1.2.7" "langchain-openai>=1.1.7" "langgraph>=1.0.7" "mistune>=3.2.1" "pymupdf4llm>=1.27.2.3" "streamlit>=1.57.0"


In [2]:
"""Implementing data synthizing for LLM Training or Fine-tuning"""

import os
import sys
import pymupdf4llm

sys.path.insert(0, os.path.abspath(".."))

In [3]:
files = [
    "/Users/vasim/Downloads/Data Engineering for Foundation Models: The Alchemist’s Cookbook | Mohammed Vasim.pdf",
    "/Users/vasim/Downloads/The Tale of Meaningful Vectors: Contrastive Learning for Text Embeddings, Told with Pen and Paper | Mohammed Vasim.pdf",
]

In [4]:
def convert_pdf_to_md(files: list[str], write_images: bool = False):
    """Convert pdf to markdown"""
    content = {}

    for file in files:
        filename = os.path.basename(file)
        img_path = f"../data/images/{filename}"
        content[filename] = pymupdf4llm.to_markdown(
            file, image_path=img_path, write_images=write_images
        )

        if write_images:
            content[img_path] = [
                os.path.join(img_path, image_path)
                for image_path in os.listdir(img_path)
            ]

    return content

In [5]:
md_texts = convert_pdf_to_md(files)
md_texts

{'Data Engineering for Foundation Models: The Alchemist’s Cookbook | Mohammed Vasim.pdf': '## Back to blog \n\n```\n~/blog\n```\n\n## **Data Engineering for Foundation Models: The Alchemist’s Cookbook** \n\n**==> picture [521 x 129] intentionally omitted <==**\n\n**----- Start of picture text -----**<br>\nMay 17, 2026 • 7 min read • By Mohammed Vasim<br>0<br>data-engineering llm foundation-models dataset-curation ai<br>**----- End of picture text -----**<br>\n\n\n## **On this page** \n\nIf you’ve ever trained a model, you know the truth: you can have the fanciest architecture and endless GPU hours, but if your data is bad, your model will be bad. Data is what gives a language model its personality, its skills, and its blind spots. Without good data, you’re just heating expensive hardware. \n\nThink of data engineering as a kind of alchemy. You start with messy, raw material from the world—text, code, conversations—and through careful steps, you transform it into something that makes a 

In [6]:
from utils import chunk_markdown_by_topic

In [7]:
list(md_texts.values())[0]

'## Back to blog \n\n```\n~/blog\n```\n\n## **Data Engineering for Foundation Models: The Alchemist’s Cookbook** \n\n**==> picture [521 x 129] intentionally omitted <==**\n\n**----- Start of picture text -----**<br>\nMay 17, 2026 • 7 min read • By Mohammed Vasim<br>0<br>data-engineering llm foundation-models dataset-curation ai<br>**----- End of picture text -----**<br>\n\n\n## **On this page** \n\nIf you’ve ever trained a model, you know the truth: you can have the fanciest architecture and endless GPU hours, but if your data is bad, your model will be bad. Data is what gives a language model its personality, its skills, and its blind spots. Without good data, you’re just heating expensive hardware. \n\nThink of data engineering as a kind of alchemy. You start with messy, raw material from the world—text, code, conversations—and through careful steps, you transform it into something that makes a model come alive. This isn’t magic; it’s method. Here’s how it works, told in plain words 

In [16]:
topic_wise = []

for text in list(md_texts.values()):
    topic_wise.extend(chunk_markdown_by_topic(text))

In [17]:
topic_wise

[{'title': 'Data Engineering for Foundation Models: The Alchemist’s Cookbook | On this page',
  'content': '==> picture [521 x 129] intentionally omitted <==\n----- Start of picture text -----<br>May 17, 2026 • 7 min read • By Mohammed Vasim<br>0<br>data-engineering llm foundation-models dataset-curation ai<br>----- End of picture text -----<br>\n\nIf you’ve ever trained a model, you know the truth: you can have the fanciest architecture and endless GPU hours, but if your data is bad, your model will be bad. Data is what gives a language model its personality, its skills, and its blind spots. Without good data, you’re just heating expensive hardware.\nThink of data engineering as a kind of alchemy. You start with messy, raw material from the world—text, code, conversations—and through careful steps, you transform it into something that makes a model come alive. This isn’t magic; it’s method. Here’s how it works, told in plain words with the actual techniques we use every day.'},
 {'tit

In [10]:
BASE_URL = "https://integrate.api.nvidia.com/v1"
API_KEY = os.getenv("NVIDIA_API_KEY")
CHAT_MODEL_NAME = "openai/gpt-oss-120b"

In [18]:
from langchain_openai import ChatOpenAI

# Replace these with real values
chat_model = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=API_KEY,
    base_url=BASE_URL
)

In [19]:
chat_model.invoke("hello")

AIMessage(content='Hello! 👋 How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 66, 'total_tokens': 101, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-a51f066166a10732', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e5a84-fc37-79a2-92ad-a0a042227266-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 66, 'output_tokens': 35, 'total_tokens': 101, 'input_token_details': {}, 'output_token_details': {}})

In [20]:
from pydantic import BaseModel, Field

class AtomicFact(BaseModel):
    """AtomicFact"""
    fact: str = Field(description="A single self-contained factual statement, declarative sentence")

class AtomicFacts(BaseModel):
    """AtomicFacts"""
    facts: list[AtomicFact] = Field(description="List of atomic facts extracted from the text chunk")

In [25]:
SYSTEM_PROMPT_EXTRACT_FACTS = """\
You are a meticulous fact extractor for building a high-quality training dataset.

Read the text and extract **meaningful, substantive facts** that represent the core domain knowledge. Focus on:

- Definitions, mechanisms, processes, categorizations, comparisons, causal relationships, and named techniques/concepts explicitly stated.
- Actionable information such as what a concept is, how it works, when to use it, what its properties are, or how it differs from another concept.

RULES:
1. Each fact must be a single, self-contained declarative sentence.
2. State the fact directly — NEVER use meta-framing like "The text says...", "The author states...", "The article mentions...", "According to...".
3. Skip trivial/metadata: dates, author names, reading time, picture dimensions, page structure, layout information, navigation elements.
4. Skip redundant restatements of the same idea. If two sentences convey the same core fact, keep only one.
5. Do not add, infer, or speculate beyond what is explicitly stated.
6. Extract only facts that would be useful as ground-truth context for answering a question about this topic."""

USER_PROMPT_TEMPLATE = """\
Title: {title}

Content:
{content}
"""

In [26]:
chain = chat_model.with_structured_output(AtomicFacts)

def extract_facts(chunk: dict) -> list[str]:
    """Extract facts"""
    response = chain.invoke([
        {"role": "system", "content": SYSTEM_PROMPT_EXTRACT_FACTS},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.format(
            title=chunk["title"], content=chunk["content"]
        )}
    ])
    return [f.fact for f in response.facts]

In [27]:
facts1 = extract_facts(topic_wise[0])
facts1

['If data is bad, the resulting model will be bad, regardless of how fancy the architecture is or how many GPU hours are used.',
 'Data provides a language model with its personality, its skills, and its blind spots.',
 'Training without good data only consumes expensive hardware without improving model performance.',
 'Data engineering converts messy, raw material such as text, code, and conversations into refined data that enables a model to function effectively.',
 'Data engineering is a methodical process rather than a magical one.']

In [30]:
from tqdm import tqdm

In [37]:
golden_facts = []

for topic in tqdm(topic_wise[:5], desc="Generating golden facts..."):
    golden_facts.append(extract_facts(topic))

Generating golden facts...: 100%|██████████| 5/5 [00:18<00:00,  3.62s/it]


In [38]:
golden_facts

[["Data quality determines a model's performance, so even with advanced architecture and extensive GPU resources, bad data results in a bad model.",
  "A language model's personality, skills, and blind spots are derived from the data it is trained on.",
  'Training on poor data wastes computational resources, effectively only heating expensive hardware without producing a useful model.',
  'Data engineering transforms raw, messy material such as text, code, and conversations into a refined dataset that enables a model to function effectively.'],
 ['When building a chatbot, conversation data is required.',
  'To enable a model to use tools such as a calculator or search engine, tool‑use data is needed, where each example shows the model calling an API and receiving a result.',
  'Single‑turn data consists of one question and one answer.',
  'Multi‑turn data teaches a model to handle back‑and‑forth conversations.',
  'Raw training data can be sourced from web crawls, code repositories, b

## Stage 3: Generating QA Pairs

In [32]:
class AlpacaQAPair(BaseModel):
    instruction: str = Field(description="The question / user instruction")
    input: str = Field(description="Optional context (empty string for single-turn)")
    output: str = Field(description="The answer grounded in the provided facts")

class AlpacaQAPairs(BaseModel):
    pairs: list[AlpacaQAPair] = Field(description="List of diverse instruction-response pairs")

In [41]:
SYSTEM_PROMPT_GENERATE_QA = """\
You are an expert at creating diverse, high-quality instruction data.

Given a list of atomic facts, generate a set of diverse instruction-response pairs.

Requirements:
- Each question must be answerable entirely from the provided facts — do not use external knowledge.
- Vary question types across the set: include "what", "how", "why", "compare/contrast", "list/enumerate", "explain", "define", and "true/false" or "yes/no" questions.
- Vary difficulty: some questions should be simple fact retrieval (one fact), others should require synthesizing 2-3 facts together.
- Each answer must be **200–300 words long** and explain the concept thoroughly: define it, provide context, include supporting details, and connect related facts — all strictly from the provided facts.
- Do not pad with filler or repetition. Every sentence should add substantive information.
- State answers directly without meta-framing like "Based on the facts...".
- Do NOT generate questions about the text itself (e.g. "What does the article say about X?"). Generate questions a real user would ask.
- Output each pair with instruction (the question), input (empty string), and output (the answer)."""

In [42]:
def generate_qa_pairs(facts: list[str], llm_chain) -> list[dict]:
    """Takes a list of atomic fact strings, returns Alpaca-format Q&A dicts."""
    facts_text = "\n".join(f"- {f}" for f in facts)
    resp = llm_chain.invoke([
        {"role": "system", "content": SYSTEM_PROMPT_GENERATE_QA},
        {"role": "user", "content": f"Atomic facts:\n\n{facts_text}"}
    ])
    return [p.model_dump() for p in resp.pairs]

In [43]:
qa_chain = chat_model.with_structured_output(AlpacaQAPairs)

In [44]:
all_pairs = []
for chunk_facts in tqdm(golden_facts, desc="Generating QA pairs..."):  # list[list[str]] from Stage 2
    facts_text = "\n".join(f"- {f}" for f in chunk_facts)
    resp = qa_chain.invoke([
        {"role": "system", "content": SYSTEM_PROMPT_GENERATE_QA},
        {"role": "user", "content": f"Atomic facts:\n\n{facts_text}"}
    ])
    all_pairs.extend(p.model_dump() for p in resp.pairs)

In [45]:
all_pairs

[{'instruction': "What is the relationship between data quality and a language model's performance?",
  'input': '',
  'output': "Data quality is the primary determinant of a language model's performance. When the dataset contains accurate, relevant, and well‑structured information, the model learns patterns that translate into reliable predictions and useful outputs. Conversely, if the data are noisy, inaccurate, or misaligned with the intended tasks, the model inherits those deficiencies, resulting in poor performance regardless of how sophisticated the architecture is or how many GPUs are deployed. The direct link between data quality and performance explains why models trained on high‑quality corpora consistently outperform those built on inferior material. This principle also extends to the model's emergent characteristics: the personality, skills, and blind spots that appear during inference are all derived from the training data, meaning that low‑quality inputs produce limited a

In [46]:
import pandas as pd

# all_pairs is your list of dicts: [{"instruction": ..., "input": ..., "output": ...}, ...]
df = pd.DataFrame(all_pairs)
df.head()

,instruction,input,output
0,What is the relationship between data quality ...,,Data quality is the primary determinant of a l...
1,How does training on poor data affect computat...,,Training on poor data consumes computational r...
2,"Why does a language model's personality, skill...",,"A language model internalizes patterns, facts,..."
3,List the key transformations that data enginee...,,Data engineering applies a series of transform...
4,Compare the outcomes of training a language mo...,,Training a language model with high‑quality da...


In [47]:
from datetime import datetime

In [50]:
def get_datetime():
    """Get datetime"""
    return datetime.now().strftime("%Y%M%d-%H%m%S")

In [51]:
# CSV
df.to_csv(f"../data/synthetic_dataset-{get_datetime()}.csv", index=False)

# JSON (records format — one object per line, or pretty-printed)
df.to_json(f"../data/synthetic_dataset-{get_datetime()}.json", orient="records", indent=2)